| | |
|---|---|
| **Project** | GenAI 03 -- Email Reply Draft Assistant |
| **Goal** | Classify email intent and draft reply options in different tones |
| **Stack** | sentence-transformers (all-MiniLM-L6-v2), NumPy |
| **Key Patterns** | Intent classification, tone control, prompt templates, structured outputs |

## Project Overview

This notebook builds an **email reply draft assistant** that:

1. **Classifies** incoming email intent (complaint, inquiry, meeting request, follow-up, feedback)
2. **Detects** urgency level (low/medium/high)
3. **Drafts reply options** in 3 tones (formal, friendly, empathetic)
4. **Evaluates** classification accuracy against ground truth

The system uses a two-stage pipeline:
- **Stage 1**: Semantic intent classification via embedding similarity to intent prototypes
- **Stage 2**: Template-based reply generation with tone control

This mirrors real-world email automation used by Gmail Smart Reply, Zendesk, and Salesforce Einstein.

## Learning Goals

1. Understand how to classify email intent using embedding similarity to labeled prototypes
2. Build a two-stage pipeline: classify first, then use classification to guide generation
3. Design prompt templates with tone control for different audience and situations
4. Implement structured output: typed classification results with confidence scores
5. Evaluate classification accuracy and analyze failure modes
6. Understand the tradeoffs between rule-based templates and LLM generation

## Problem Statement

You receive dozens of emails daily -- complaints, inquiries, meeting requests, follow-ups.
Each needs a different tone and level of detail. Writing replies is time-consuming and repetitive.

**Goal**: Given an incoming email, automatically:
- Classify what it is about (intent)
- Assess urgency
- Suggest the right tone
- Generate draft replies in multiple tones so the user can pick and edit

**Why two stages?** A single-prompt approach works for simple cases but fails when:
- You need the classification to be reusable (e.g., for routing, prioritization)
- You want consistent structured output
- You need multiple output variations with different tones

## Why This Project Matters

Email automation is one of the highest-ROI GenAI applications in enterprise:

| Pattern | Used Here | Also Used In |
|---------|----------|-------------|
| Intent classification | Email categorization | Ticket routing, chatbot NLU, support triage |
| Structured output | Typed classification results | API backends, data extraction, form filling |
| Two-stage pipeline | Classify then generate | Triage then route, analyze then act |
| Tone control | Formal/friendly/empathetic | Brand voice, audience targeting |
| Multi-option generation | 3 reply drafts | A/B testing, human-in-the-loop review |

## Dataset Overview

We use **manually crafted sample emails** representing common business email categories.
Using hand-crafted examples with known ground truth lets us:
- Control quality and variety
- Evaluate classification accuracy precisely
- Demonstrate all intent categories clearly

| Intent | Example Scenario | Typical Tone |
|--------|-----------------|-------------|
| Complaint | Defective product, poor service | Empathetic |
| Inquiry | Pricing question, feature request | Professional |
| Meeting request | Schedule alignment | Friendly |
| Follow-up | Checking status of a prior request | Professional |
| Feedback | Positive or negative product review | Appreciative |

## Environment Setup

In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                        'sentence-transformers', 'numpy'])
print('Dependencies ready.')

Dependencies ready.


## Imports

In [2]:
import numpy as np
import time
import json
import textwrap
import warnings
from sentence_transformers import SentenceTransformer
from collections import defaultdict
warnings.filterwarnings('ignore')
print('Imports complete.')

E:\Github\Machine-Learning-Projects\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


E:\Github\Machine-Learning-Projects\.venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.7) doesn't match a supported version!
  warnings.warn(


Imports complete.


## Configuration

In [3]:
MODEL_NAME = 'all-MiniLM-L6-v2'
SEED = 42
np.random.seed(SEED)

# Intent categories
INTENT_CATEGORIES = ['complaint', 'inquiry', 'meeting_request', 'follow_up', 'feedback']

# Urgency keywords
HIGH_URGENCY_KEYWORDS = ['asap', 'urgent', 'immediately', 'critical', 'emergency', 'deadline',
                          'frustrated', 'unacceptable', 'defective', 'broken']
LOW_URGENCY_KEYWORDS = ['whenever', 'no rush', 'at your convenience', 'fyi', 'just wanted to share']

print(f'Model: {MODEL_NAME}')
print(f'Intent categories: {INTENT_CATEGORIES}')

Model: all-MiniLM-L6-v2
Intent categories: ['complaint', 'inquiry', 'meeting_request', 'follow_up', 'feedback']


## Load Embedding Model

In [4]:
t0 = time.time()
model = SentenceTransformer(MODEL_NAME)
load_time = time.time() - t0
print(f'Model loaded in {load_time:.1f}s')
print(f'Embedding dim: {model.get_sentence_embedding_dimension()}')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10930.68it/s]


BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded in 6.7s
Embedding dim: 384


## Sample Email Dataset

Each email has a known ground-truth intent and urgency for evaluation.

In [5]:
SAMPLE_EMAILS = [
    # --- Complaints ---
    {
        'id': 'E01', 'expected_intent': 'complaint', 'expected_urgency': 'high',
        'subject': 'Extremely disappointed with recent order #4582',
        'body': '''I ordered the Premium Wireless Headphones (Order #4582) two weeks ago and they arrived with a cracked case and the left ear cup does not produce any sound. This is the second defective product I have received from your store. I need an immediate replacement or a full refund. I have been a loyal customer for 3 years and this experience has been really frustrating. Please respond ASAP.\n\nThanks, Sarah Johnson'''
    },
    {
        'id': 'E02', 'expected_intent': 'complaint', 'expected_urgency': 'high',
        'subject': 'Billing error on my account',
        'body': '''I was charged twice for my subscription this month. Transaction IDs: TXN-8891 and TXN-8892, both for $49.99. This is unacceptable. I need the duplicate charge reversed immediately. I have attached my bank statement as proof.\n\nRegards, Tom Williams'''
    },

    # --- Inquiries ---
    {
        'id': 'E03', 'expected_intent': 'inquiry', 'expected_urgency': 'medium',
        'subject': 'Question about Enterprise API pricing',
        'body': '''We are evaluating your API platform for our company (500+ employees). Could you provide information on enterprise pricing tiers, SLA guarantees, and custom integration support? We would also like to know if you offer a pilot program.\n\nRegards, David Park, CTO, TechCorp'''
    },
    {
        'id': 'E04', 'expected_intent': 'inquiry', 'expected_urgency': 'low',
        'subject': 'Does your software support Linux?',
        'body': '''Hi, I am considering purchasing your design software but I primarily work on Ubuntu 22.04. Does your software support Linux natively or only via Wine? No rush on the response, just planning ahead for next quarter.\n\nBest, Alex'''
    },

    # --- Meeting Requests ---
    {
        'id': 'E05', 'expected_intent': 'meeting_request', 'expected_urgency': 'medium',
        'subject': 'Q3 Planning Session - Can we meet Thursday?',
        'body': '''I would like to schedule a Q3 planning session to discuss our roadmap priorities and resource allocation. Would Thursday at 2 PM work for everyone? Agenda: 1. Review Q2 metrics 2. Discuss Q3 OKRs 3. Resource planning. Please confirm your availability.\n\nBest, Mike Chen'''
    },
    {
        'id': 'E06', 'expected_intent': 'meeting_request', 'expected_urgency': 'medium',
        'subject': 'Demo call for new client',
        'body': '''Hi team, the new client Acme Inc wants a product demo next week. Can we schedule a 45-minute call on Tuesday or Wednesday morning? I will prepare the slides if someone can handle the live demo portion.\n\nThanks, Rachel'''
    },

    # --- Follow-ups ---
    {
        'id': 'E07', 'expected_intent': 'follow_up', 'expected_urgency': 'medium',
        'subject': 'Re: Project proposal status?',
        'body': '''Just checking in on the project proposal I submitted two weeks ago. Has the review committee had a chance to look at it? I am happy to provide any additional information or join a call to discuss. Any update would be appreciated.\n\nBest, James'''
    },
    {
        'id': 'E08', 'expected_intent': 'follow_up', 'expected_urgency': 'high',
        'body': '''This is my third email about the refund for order #3301. It has been 4 weeks since I was promised a refund within 5-7 business days. I need this resolved immediately or I will dispute the charge with my bank.\n\nSincerely, Karen White''',
        'subject': 'URGENT: Still waiting for refund - Order #3301'
    },

    # --- Feedback ---
    {
        'id': 'E09', 'expected_intent': 'feedback', 'expected_urgency': 'low',
        'subject': 'Great experience with your support team',
        'body': '''Just wanted to share that your support agent Maria was incredibly helpful when I had issues setting up the integration. She walked me through every step and followed up the next day to make sure everything was working. Fantastic service!\n\nBest, Carlos'''
    },
    {
        'id': 'E10', 'expected_intent': 'feedback', 'expected_urgency': 'low',
        'subject': 'Product suggestion for next release',
        'body': '''I have been using your app daily for 6 months. It would be great if you could add dark mode and keyboard shortcuts for power users. The current UI is good but these features would make it excellent. Just wanted to share my thoughts.\n\nCheers, Priya'''
    },
]

print(f'Sample emails: {len(SAMPLE_EMAILS)}')
for intent in INTENT_CATEGORIES:
    count = sum(1 for e in SAMPLE_EMAILS if e['expected_intent'] == intent)
    print(f'  {intent}: {count}')

Sample emails: 10
  complaint: 2
  inquiry: 2
  meeting_request: 2
  follow_up: 2
  feedback: 2


## Intent Classification via Embedding Similarity

We define **prototype descriptions** for each intent category. To classify an email,
we compute cosine similarity between the email embedding and each prototype embedding.
The highest-similarity prototype determines the intent.

This approach works because semantically similar text maps to nearby vectors.

In [6]:
# Intent prototypes: short descriptions that capture the essence of each category
INTENT_PROTOTYPES = {
    'complaint': [
        'I am very unhappy with the product and want a refund or replacement.',
        'This is unacceptable. The item arrived damaged and defective.',
        'I am frustrated with the poor service and want to file a complaint.'
    ],
    'inquiry': [
        'I have a question about your product pricing and features.',
        'Could you provide more information about your services?',
        'I would like to learn about your enterprise plans and integrations.'
    ],
    'meeting_request': [
        'Can we schedule a meeting to discuss the project?',
        'I would like to set up a call to go over the agenda.',
        'Are you available for a meeting on Thursday afternoon?'
    ],
    'follow_up': [
        'Just checking in on the status of my previous request.',
        'I wanted to follow up on the proposal I sent last week.',
        'Any update on the issue I reported earlier?'
    ],
    'feedback': [
        'I just wanted to share that your service was excellent.',
        'Here are some suggestions for improving your product.',
        'Great experience overall, but I have a few feature requests.'
    ]
}

# Encode all prototypes
proto_embeddings = {}  # intent -> avg embedding
for intent, texts in INTENT_PROTOTYPES.items():
    embs = model.encode(texts, normalize_embeddings=True, show_progress_bar=False)
    proto_embeddings[intent] = np.mean(embs, axis=0)
    # Re-normalize the averaged embedding
    proto_embeddings[intent] = proto_embeddings[intent] / np.linalg.norm(proto_embeddings[intent])

print(f'Encoded {len(proto_embeddings)} intent prototypes')
for intent in INTENT_CATEGORIES:
    print(f'  {intent}: {len(INTENT_PROTOTYPES[intent])} prototype sentences')

Encoded 5 intent prototypes
  complaint: 3 prototype sentences
  inquiry: 3 prototype sentences
  meeting_request: 3 prototype sentences
  follow_up: 3 prototype sentences
  feedback: 3 prototype sentences


## Email Intent Classifier

In [7]:
def classify_email(email):
    """Classify email intent and urgency. Returns structured output."""
    # Combine subject and body for classification
    full_text = f"{email['subject']}. {email['body']}"
    
    # Encode email
    email_emb = model.encode([full_text], normalize_embeddings=True, show_progress_bar=False)[0]
    
    # Compute similarity to each intent prototype
    scores = {}
    for intent, proto_emb in proto_embeddings.items():
        scores[intent] = float(np.dot(email_emb, proto_emb))
    
    # Best intent
    best_intent = max(scores, key=scores.get)
    confidence = scores[best_intent]
    
    # Urgency detection via keywords
    text_lower = full_text.lower()
    high_count = sum(1 for kw in HIGH_URGENCY_KEYWORDS if kw in text_lower)
    low_count = sum(1 for kw in LOW_URGENCY_KEYWORDS if kw in text_lower)
    
    if high_count >= 2:
        urgency = 'high'
    elif high_count >= 1:
        urgency = 'medium' if low_count == 0 else 'low'
    elif low_count >= 1:
        urgency = 'low'
    else:
        urgency = 'medium'
    
    # Suggested tone based on intent
    tone_map = {
        'complaint': 'empathetic',
        'inquiry': 'professional',
        'meeting_request': 'friendly',
        'follow_up': 'professional',
        'feedback': 'appreciative'
    }
    suggested_tone = tone_map.get(best_intent, 'professional')
    
    # Structured output
    return {
        'intent': best_intent,
        'confidence': confidence,
        'urgency': urgency,
        'suggested_tone': suggested_tone,
        'all_scores': scores
    }

# Quick test
test_result = classify_email(SAMPLE_EMAILS[0])
print(f'Test email: {SAMPLE_EMAILS[0]["subject"]}')
print(f'Classified: {json.dumps(test_result, indent=2, default=str)}')

Test email: Extremely disappointed with recent order #4582
Classified: {
  "intent": "complaint",
  "confidence": 0.4072030782699585,
  "urgency": "high",
  "suggested_tone": "empathetic",
  "all_scores": {
    "complaint": 0.4072030782699585,
    "inquiry": 0.13991284370422363,
    "meeting_request": 0.023229029029607773,
    "follow_up": 0.21245068311691284,
    "feedback": 0.20677845180034637
  }
}


## Prompt Templates and Structured Outputs

### What are Prompt Templates?

Prompt templates are parameterized text structures that guide language model output.
Instead of ad-hoc prompts, templates provide:
- **Consistency**: Same structure for every input
- **Reusability**: Change variables, keep the structure
- **Quality control**: Explicit format instructions reduce hallucination

### Why Structured Output Matters

Without structured output, you get free-form text that you must regex-parse (brittle).
With structured output (typed fields + format constraints), you get:

```python
# BAD: Free-form
"The email is about a complaint. It seems urgent."

# GOOD: Structured
{'intent': 'complaint', 'urgency': 'high', 'confidence': 0.87}
```

Our classifier already returns structured output. The reply templates below
use the classification to select the right reply strategy.

## Reply Draft Templates with Tone Control

In [8]:
# Reply templates organized by intent and tone
REPLY_TEMPLATES = {
    'complaint': {
        'formal': (
            'Dear {sender},\n\n'
            'Thank you for bringing this matter to our attention regarding your {subject_ref}. '
            'We sincerely apologize for the inconvenience you have experienced. '
            'We take such concerns very seriously and have escalated your case to our resolution team.\n\n'
            'You can expect a follow-up within 24 hours with a concrete resolution. '
            'Your case reference number is {case_ref}.\n\n'
            'Respectfully,\n{company} Support Team'
        ),
        'friendly': (
            'Hi {sender},\n\n'
            'Thanks so much for reaching out about this -- and I am really sorry about what happened '
            'with your {subject_ref}. That is definitely not the experience we want you to have!\n\n'
            'I have flagged this for our team and someone will be in touch within 24 hours to make this right. '
            'Your case number is {case_ref}.\n\n'
            'Thanks for your patience!\nThe {company} Team'
        ),
        'empathetic': (
            'Dear {sender},\n\n'
            'I completely understand how frustrating this must be, especially regarding your {subject_ref}. '
            'You deserve better and I am personally sorry for this experience.\n\n'
            'I want to make sure we resolve this for you as quickly as possible. '
            'I have escalated your case (ref: {case_ref}) to our senior resolution team, '
            'and you will hear back within 24 hours with a solution.\n\n'
            'Thank you for your patience and loyalty.\nBest regards,\n{company} Support'
        )
    },
    'inquiry': {
        'formal': (
            'Dear {sender},\n\n'
            'Thank you for your interest in our services. Regarding your inquiry about {subject_ref}, '
            'I have compiled the relevant information below.\n\n'
            'Our team would be happy to schedule a detailed consultation at your convenience. '
            'Please let us know your preferred time and we will arrange a call.\n\n'
            'Best regards,\n{company} Sales Team'
        ),
        'friendly': (
            'Hi {sender},\n\n'
            'Great to hear from you! Thanks for asking about {subject_ref}.\n\n'
            'I would love to walk you through the details -- would a quick call work? '
            'We can cover everything and answer any other questions you might have.\n\n'
            'Looking forward to chatting!\nThe {company} Team'
        ),
        'empathetic': (
            'Dear {sender},\n\n'
            'Thank you for reaching out about {subject_ref}. '
            'I understand that choosing the right solution is an important decision, '
            'and I want to make sure you have all the information you need.\n\n'
            'I am happy to set up a personalized walkthrough at a time that works for you.\n\n'
            'Warm regards,\n{company} Team'
        )
    },
    'meeting_request': {
        'formal': (
            'Dear {sender},\n\n'
            'Thank you for proposing the meeting regarding {subject_ref}. '
            'I have reviewed the agenda and confirm my availability. '
            'Please send a calendar invitation at your earliest convenience.\n\n'
            'Best regards,\n{company}'
        ),
        'friendly': (
            'Hi {sender},\n\n'
            'Sounds great! I am available for the {subject_ref} meeting. '
            'Count me in -- looking forward to the discussion!\n\n'
            'Cheers,\n{company}'
        ),
        'empathetic': (
            'Hi {sender},\n\n'
            'Thanks for organizing this -- I know coordinating schedules can be tricky! '
            'I am happy to join the discussion about {subject_ref}. '
            'The proposed time works well for me.\n\n'
            'Looking forward to it,\n{company}'
        )
    },
    'follow_up': {
        'formal': (
            'Dear {sender},\n\n'
            'Thank you for following up regarding {subject_ref}. '
            'I apologize for the delay in our response. '
            'Your request is currently under review and I will provide a comprehensive update '
            'by end of business tomorrow.\n\n'
            'Best regards,\n{company}'
        ),
        'friendly': (
            'Hi {sender},\n\n'
            'Thanks for checking in on {subject_ref}! '
            'Sorry for the wait -- let me get you an update. '
            'I will have more details for you by tomorrow.\n\n'
            'Thanks for your patience!\n{company}'
        ),
        'empathetic': (
            'Dear {sender},\n\n'
            'I appreciate you reaching out again about {subject_ref}, and I completely '
            'understand your concern about the timeline. I want to assure you that your request '
            'has not been forgotten. I am personally looking into this and will get back to you '
            'with a detailed update by tomorrow.\n\n'
            'Thank you for your patience,\n{company}'
        )
    },
    'feedback': {
        'formal': (
            'Dear {sender},\n\n'
            'Thank you for taking the time to share your feedback regarding {subject_ref}. '
            'We value input from our users and your suggestions have been forwarded to our product team '
            'for consideration in future releases.\n\n'
            'Best regards,\n{company}'
        ),
        'friendly': (
            'Hi {sender},\n\n'
            'Thanks so much for sharing your thoughts on {subject_ref}! '
            'We really appreciate hearing from users like you. '
            'I have passed your suggestions to our product team -- they love getting this kind of input!\n\n'
            'Cheers,\nThe {company} Team'
        ),
        'empathetic': (
            'Dear {sender},\n\n'
            'Thank you so much for your thoughtful feedback about {subject_ref}. '
            'It means a lot to us that you took the time to share your experience. '
            'Your suggestions are exactly what helps us improve, and I have personally '
            'shared them with our product team.\n\n'
            'Warmly,\n{company}'
        )
    }
}

print(f'Reply templates: {len(REPLY_TEMPLATES)} intents x 3 tones = {len(REPLY_TEMPLATES) * 3} templates')

Reply templates: 5 intents x 3 tones = 15 templates


## Reply Generation Pipeline

In [9]:
def extract_sender(email):
    """Extract sender name from email body."""
    body = email['body']
    # Look for common sign-off patterns
    for marker in ['Thanks, ', 'Regards, ', 'Best, ', 'Cheers, ', 'Sincerely, ',
                   'Best regards, ', 'Warm regards, ']:
        if marker in body:
            after = body.split(marker)[-1].strip()
            name = after.split('\n')[0].strip()
            if name and len(name) < 40:
                return name
    return 'Valued Customer'

def generate_replies(email, classification, company='Acme Corp'):
    """Generate reply drafts in 3 tones based on classification."""
    intent = classification['intent']
    sender = extract_sender(email)
    subject_ref = email['subject'].lower().replace('re: ', '').replace('urgent: ', '')
    case_ref = f'CASE-{hash(email["id"]) % 10000:04d}'
    
    templates = REPLY_TEMPLATES.get(intent, REPLY_TEMPLATES['inquiry'])
    
    replies = {}
    for tone, template in templates.items():
        reply = template.format(
            sender=sender,
            subject_ref=subject_ref,
            case_ref=case_ref,
            company=company
        )
        replies[tone] = reply
    
    return replies

# Quick test
test_class = classify_email(SAMPLE_EMAILS[0])
test_replies = generate_replies(SAMPLE_EMAILS[0], test_class)
print(f'Email: {SAMPLE_EMAILS[0]["subject"]}')
print(f'Intent: {test_class["intent"]}, Urgency: {test_class["urgency"]}')
print(f'Suggested tone: {test_class["suggested_tone"]}')
print(f'\n--- Suggested reply ({test_class["suggested_tone"]}) ---')
print(test_replies[test_class['suggested_tone']])

Email: Extremely disappointed with recent order #4582
Intent: complaint, Urgency: high
Suggested tone: empathetic

--- Suggested reply (empathetic) ---
Dear Sarah Johnson,

I completely understand how frustrating this must be, especially regarding your extremely disappointed with recent order #4582. You deserve better and I am personally sorry for this experience.

I want to make sure we resolve this for you as quickly as possible. I have escalated your case (ref: CASE-0760) to our senior resolution team, and you will hear back within 24 hours with a solution.

Thank you for your patience and loyalty.
Best regards,
Acme Corp Support


## Run Full Pipeline on All Emails

In [10]:
all_results = []

print('Email Classification and Reply Generation')
print('=' * 85)

for email in SAMPLE_EMAILS:
    t0 = time.time()
    classification = classify_email(email)
    replies = generate_replies(email, classification)
    latency_ms = (time.time() - t0) * 1000
    
    # Check accuracy
    intent_correct = classification['intent'] == email['expected_intent']
    urgency_correct = classification['urgency'] == email['expected_urgency']
    
    result = {
        'id': email['id'],
        'subject': email['subject'],
        'expected_intent': email['expected_intent'],
        'predicted_intent': classification['intent'],
        'intent_correct': intent_correct,
        'confidence': classification['confidence'],
        'expected_urgency': email['expected_urgency'],
        'predicted_urgency': classification['urgency'],
        'urgency_correct': urgency_correct,
        'suggested_tone': classification['suggested_tone'],
        'latency_ms': latency_ms,
        'replies': replies,
        'all_scores': classification['all_scores']
    }
    all_results.append(result)
    
    status = 'OK' if intent_correct else 'MISS'
    print(f'{email["id"]} [{status}] intent={classification["intent"]:16s} '
          f'(exp={email["expected_intent"]:16s}) conf={classification["confidence"]:.3f} '
          f'urgency={classification["urgency"]:6s} | {email["subject"][:45]}')

print('\n' + '=' * 85)
print('Pipeline complete.')

Email Classification and Reply Generation
E01 [OK] intent=complaint        (exp=complaint       ) conf=0.407 urgency=high   | Extremely disappointed with recent order #458
E02 [OK] intent=complaint        (exp=complaint       ) conf=0.327 urgency=high   | Billing error on my account
E03 [OK] intent=inquiry          (exp=inquiry         ) conf=0.630 urgency=medium | Question about Enterprise API pricing
E04 [OK] intent=inquiry          (exp=inquiry         ) conf=0.240 urgency=low    | Does your software support Linux?
E05 [OK] intent=meeting_request  (exp=meeting_request ) conf=0.661 urgency=medium | Q3 Planning Session - Can we meet Thursday?
E06 [OK] intent=meeting_request  (exp=meeting_request ) conf=0.577 urgency=medium | Demo call for new client
E07 [OK] intent=follow_up        (exp=follow_up       ) conf=0.522 urgency=medium | Re: Project proposal status?
E08 [MISS] intent=complaint        (exp=follow_up       ) conf=0.455 urgency=high   | URGENT: Still waiting for refund - Order

## Classification Accuracy

In [11]:
n = len(all_results)
intent_correct = sum(1 for r in all_results if r['intent_correct'])
urgency_correct = sum(1 for r in all_results if r['urgency_correct'])
avg_confidence = float(np.mean([r['confidence'] for r in all_results]))
avg_latency = float(np.mean([r['latency_ms'] for r in all_results]))

print('CLASSIFICATION ACCURACY')
print('=' * 45)
print(f'Intent accuracy:    {intent_correct}/{n} = {intent_correct/n:.1%}')
print(f'Urgency accuracy:   {urgency_correct}/{n} = {urgency_correct/n:.1%}')
print(f'Avg confidence:     {avg_confidence:.4f}')
print(f'Avg latency:        {avg_latency:.1f}ms')

# Per-intent accuracy
print('\nPer-intent breakdown:')
for intent in INTENT_CATEGORIES:
    subset = [r for r in all_results if r['expected_intent'] == intent]
    if not subset:
        continue
    correct = sum(1 for r in subset if r['intent_correct'])
    avg_conf = float(np.mean([r['confidence'] for r in subset]))
    print(f'  {intent:16s}: {correct}/{len(subset)} correct, avg_conf={avg_conf:.4f}')

CLASSIFICATION ACCURACY
Intent accuracy:    9/10 = 90.0%
Urgency accuracy:   10/10 = 100.0%
Avg confidence:     0.4648
Avg latency:        7.0ms

Per-intent breakdown:
  complaint       : 2/2 correct, avg_conf=0.3670
  inquiry         : 2/2 correct, avg_conf=0.4351
  meeting_request : 2/2 correct, avg_conf=0.6193
  follow_up       : 1/2 correct, avg_conf=0.4887
  feedback        : 2/2 correct, avg_conf=0.4138


## Intent Classification Confusion Analysis

In [12]:
# Build confusion data
print('Confusion Matrix:')
print(f'{"Predicted ->":>18s}', end='')
for intent in INTENT_CATEGORIES:
    print(f' {intent[:8]:>8s}', end='')
print()
print('-' * (18 + 9 * len(INTENT_CATEGORIES)))

for true_intent in INTENT_CATEGORIES:
    print(f'{true_intent:>18s}', end='')
    for pred_intent in INTENT_CATEGORIES:
        count = sum(1 for r in all_results
                    if r['expected_intent'] == true_intent and r['predicted_intent'] == pred_intent)
        print(f' {count:>8d}', end='')
    print()

# Misclassified emails
misses = [r for r in all_results if not r['intent_correct']]
if misses:
    print(f'\nMisclassified emails ({len(misses)}):')
    for r in misses:
        print(f'  {r["id"]}: expected={r["expected_intent"]}, predicted={r["predicted_intent"]} '
              f'(conf={r["confidence"]:.3f})')
        # Show score gap
        expected_score = r['all_scores'][r['expected_intent']]
        print(f'    Score for expected: {expected_score:.4f} vs predicted: {r["confidence"]:.4f} '
              f'(gap={r["confidence"]-expected_score:.4f})')
else:
    print('\nAll emails classified correctly!')

Confusion Matrix:
      Predicted -> complain  inquiry meeting_ follow_u feedback
---------------------------------------------------------------
         complaint        2        0        0        0        0
           inquiry        0        2        0        0        0
   meeting_request        0        0        2        0        0
         follow_up        1        0        0        1        0
          feedback        0        0        0        0        2

Misclassified emails (1):
  E08: expected=follow_up, predicted=complaint (conf=0.455)
    Score for expected: 0.3751 vs predicted: 0.4554 (gap=0.0804)


## Per-Email Detailed Results

In [13]:
print(f'{"ID":>4s} | {"Expected":>16s} | {"Predicted":>16s} | {"Conf":>6s} | {"Urg":>6s} | {"Tone":>12s} | {"ms":>6s} | Subject')
print('-' * 110)
for r in all_results:
    mark = 'OK' if r['intent_correct'] else 'XX'
    print(f'{r["id"]:>4s} | {r["expected_intent"]:>16s} | {r["predicted_intent"]:>16s} | '
          f'{r["confidence"]:>6.3f} | {r["predicted_urgency"]:>6s} | {r["suggested_tone"]:>12s} | '
          f'{r["latency_ms"]:>6.0f} | [{mark}] {r["subject"][:35]}')

  ID |         Expected |        Predicted |   Conf |    Urg |         Tone |     ms | Subject
--------------------------------------------------------------------------------------------------------------
 E01 |        complaint |        complaint |  0.407 |   high |   empathetic |      8 | [OK] Extremely disappointed with recent 
 E02 |        complaint |        complaint |  0.327 |   high |   empathetic |      7 | [OK] Billing error on my account
 E03 |          inquiry |          inquiry |  0.630 | medium | professional |      7 | [OK] Question about Enterprise API prici
 E04 |          inquiry |          inquiry |  0.240 |    low | professional |      7 | [OK] Does your software support Linux?
 E05 |  meeting_request |  meeting_request |  0.661 | medium |     friendly |      7 | [OK] Q3 Planning Session - Can we meet T
 E06 |  meeting_request |  meeting_request |  0.577 | medium |     friendly |      7 | [OK] Demo call for new client
 E07 |        follow_up |        follow_up |  0

## Tone Comparison: Same Email, Three Tones

We show how the same email gets different reply drafts depending on tone.
This demonstrates the value of tone control in production email assistants.

In [14]:
# Show all 3 tones for a complaint and an inquiry
for idx in [0, 2]:  # E01 (complaint), E03 (inquiry)
    email = SAMPLE_EMAILS[idx]
    result = all_results[idx]
    print(f'\n{"=" * 80}')
    print(f'EMAIL: {email["subject"]}')
    print(f'Intent: {result["predicted_intent"]} | Urgency: {result["predicted_urgency"]} | Suggested tone: {result["suggested_tone"]}')
    
    for tone in ['formal', 'friendly', 'empathetic']:
        marker = ' [SUGGESTED]' if tone == result['suggested_tone'] else ''
        print(f'\n--- {tone.upper()} TONE{marker} ---')
        print(result['replies'][tone])


EMAIL: Extremely disappointed with recent order #4582
Intent: complaint | Urgency: high | Suggested tone: empathetic

--- FORMAL TONE ---
Dear Sarah Johnson,

Thank you for bringing this matter to our attention regarding your extremely disappointed with recent order #4582. We sincerely apologize for the inconvenience you have experienced. We take such concerns very seriously and have escalated your case to our resolution team.

You can expect a follow-up within 24 hours with a concrete resolution. Your case reference number is CASE-0760.

Respectfully,
Acme Corp Support Team

--- FRIENDLY TONE ---
Hi Sarah Johnson,

Thanks so much for reaching out about this -- and I am really sorry about what happened with your extremely disappointed with recent order #4582. That is definitely not the experience we want you to have!

I have flagged this for our team and someone will be in touch within 24 hours to make this right. Your case number is CASE-0760.

Thanks for your patience!
The Acme Corp

## Intent Score Distribution Analysis

In [15]:
print('Per-email intent scores (higher = more likely):')
print(f'{"ID":>4s} | {"Expected":>16s}', end='')
for intent in INTENT_CATEGORIES:
    print(f' | {intent[:8]:>8s}', end='')
print()
print('-' * (25 + 11 * len(INTENT_CATEGORIES)))

for r in all_results:
    print(f'{r["id"]:>4s} | {r["expected_intent"]:>16s}', end='')
    for intent in INTENT_CATEGORIES:
        score = r['all_scores'][intent]
        marker = '*' if intent == r['predicted_intent'] else ' '
        print(f' | {score:>7.4f}{marker}', end='')
    print()

Per-email intent scores (higher = more likely):
  ID |         Expected | complain |  inquiry | meeting_ | follow_u | feedback
--------------------------------------------------------------------------------
 E01 |        complaint |  0.4072* |  0.1399  |  0.0232  |  0.2125  |  0.2068 
 E02 |        complaint |  0.3268* |  0.1405  |  0.0399  |  0.2378  |  0.0874 
 E03 |          inquiry |  0.0977  |  0.6298* |  0.1928  |  0.1748  |  0.3140 
 E04 |          inquiry |  0.0321  |  0.2404* |  0.1084  |  0.1521  |  0.1973 
 E05 |  meeting_request |  0.0284  |  0.2962  |  0.6614* |  0.2529  |  0.1920 
 E06 |  meeting_request |  0.1766  |  0.2595  |  0.5771* |  0.1954  |  0.1754 
 E07 |        follow_up |  0.0890  |  0.2074  |  0.4294  |  0.5219* |  0.2366 
 E08 |        follow_up |  0.4554* |  0.1763  |  0.1653  |  0.3751  |  0.1808 
 E09 |         feedback |  0.2407  |  0.3717  |  0.1616  |  0.4296  |  0.4428*
 E10 |         feedback |  0.1133  |  0.2311  |  0.1169  |  0.2329  |  0.3848*


## Error Analysis

In [16]:
print('Error Analysis')
print('=' * 60)

# Intent misses
print(f'\nIntent accuracy: {intent_correct}/{n} ({intent_correct/n:.0%})')
if misses:
    for r in misses:
        print(f'  MISS {r["id"]}: {r["subject"][:50]}')
        print(f'    Expected: {r["expected_intent"]}, Got: {r["predicted_intent"]}')
        print(f'    Root cause: score gap is only {abs(r["confidence"] - r["all_scores"][r["expected_intent"]]):.4f}')

# Urgency misses
urg_misses = [r for r in all_results if not r['urgency_correct']]
print(f'\nUrgency accuracy: {urgency_correct}/{n} ({urgency_correct/n:.0%})')
for r in urg_misses:
    print(f'  MISS {r["id"]}: expected={r["expected_urgency"]}, got={r["predicted_urgency"]}')
    print(f'    Subject: {r["subject"][:50]}')

# Confidence analysis
correct_confs = [r['confidence'] for r in all_results if r['intent_correct']]
wrong_confs = [r['confidence'] for r in all_results if not r['intent_correct']]
print(f'\nConfidence for correct predictions: {np.mean(correct_confs):.4f}' if correct_confs else '')
print(f'Confidence for wrong predictions:   {np.mean(wrong_confs):.4f}' if wrong_confs else 'No wrong predictions.')

# Hardest intent (lowest avg confidence)
print('\nHardest intent (lowest avg margin):')
for intent in INTENT_CATEGORIES:
    subset = [r for r in all_results if r['expected_intent'] == intent]
    if subset:
        margins = []
        for r in subset:
            expected_score = r['all_scores'][intent]
            other_scores = [v for k, v in r['all_scores'].items() if k != intent]
            margin = expected_score - max(other_scores)
            margins.append(margin)
        avg_margin = float(np.mean(margins))
        print(f'  {intent:16s}: avg margin = {avg_margin:.4f}')

Error Analysis

Intent accuracy: 9/10 (90%)
  MISS E08: URGENT: Still waiting for refund - Order #3301
    Expected: follow_up, Got: complaint
    Root cause: score gap is only 0.0804

Urgency accuracy: 10/10 (100%)

Confidence for correct predictions: 0.4658
Confidence for wrong predictions:   0.4554

Hardest intent (lowest avg margin):
  complaint       : avg margin = 0.1419
  inquiry         : avg margin = 0.1794
  meeting_request : avg margin = 0.3414
  follow_up       : avg margin = 0.0061
  feedback        : avg margin = 0.0826


## How Prompt Templates Work

### Template Structure

A good prompt template has these components:

```
1. ROLE:      "You are a customer support agent..."
2. CONTEXT:   "The email is classified as {intent} with {urgency} urgency"
3. INPUT:     "Email: {email_text}"
4. FORMAT:    "Reply in {tone} tone. Include: greeting, acknowledgment, action, sign-off"
5. EXAMPLES:  (optional) "Example reply: ..."
6. CONSTRAINTS: "Keep under 150 words. Do not promise specific timelines."
```

### Why Templates Beat Free-Form Prompts

| Approach | Reliability | Structure | Reusability |
|----------|------------|-----------|-------------|
| "Reply to this email" | Low | None | No |
| Template with role + format | High | Consistent | Yes |
| Template + structured output | Highest | Typed fields | Yes |

### Tone Control Mechanism

Our templates use tone as a parameter to select different phrasings:
- **Formal**: "Dear", "Respectfully", "Best regards"
- **Friendly**: "Hi", "Thanks so much", "Cheers"
- **Empathetic**: "I completely understand", "I am personally sorry", "Warmly"

In production with an LLM, tone would be a prompt parameter:
```
Write a {tone} reply to the following email...
```

## Limitations

1. **Template-based replies** lack the fluency and personalization of LLM-generated responses -- they cannot reference specific details from the email body
2. **Intent prototypes** use averaged embeddings from 3 sentences per category -- more prototypes would improve accuracy
3. **Keyword-based urgency** misses nuanced urgency signals (tone, context, sender importance)
4. **5 intent categories** are a simplification -- real email systems need 20+ categories including spam, out-of-office, newsletters
5. **No sender/context awareness** -- the system does not consider sender history, relationship, or organizational context
6. **Small test set** (10 emails) -- production evaluation needs hundreds of labeled examples

## Common Mistakes

| Mistake | Why It's Wrong | What To Do Instead |
|---------|---------------|--------------------|
| Single-prompt "reply to this email" | No structured output, inconsistent quality | Two-stage: classify then generate |
| Hardcoded intent rules | Brittle, fails on novel phrasings | Use embedding similarity with prototypes |
| Same tone for all emails | Complaints need empathy, not cheerfulness | Map intent to suggested tone |
| No confidence score | Cannot detect uncertain classifications | Report similarity scores and set thresholds |
| Regex parsing LLM output | Breaks when format changes | Use structured output (Pydantic, JSON schema) |
| Ignoring urgency | High-urgency emails treated same as low | Detect urgency and prioritize accordingly |

## Mini Challenge

1. Add 3 more intent categories (out-of-office, spam, introduction) with appropriate prototypes and reply templates
2. Improve urgency detection by adding embedding-based urgency scoring alongside keyword matching
3. Make reply templates reference specific details from the email body (e.g., order numbers, product names)
4. Replace template-based generation with an LLM (e.g., Qwen3-Instruct) that uses the classification as context
5. Build a feedback loop: let users rate reply quality and use ratings to improve prototype selection

## Production Considerations

| Area | Recommendation |
|------|---------------|
| Classification | Use fine-tuned classifier (ModernBERT) instead of prototype similarity for 20+ categories |
| Reply generation | Use Qwen3-Instruct / GPT-4o with structured output for personalized replies |
| Tone detection | Detect email tone from the incoming message to auto-match reply tone |
| Confidence threshold | Set minimum confidence (e.g., 0.5) and route uncertain emails to humans |
| Safety | Add content filters to prevent generating inappropriate replies |
| Personalization | Include sender history, CRM data, and organizational context |
| A/B testing | Test which reply tones get better response rates per intent category |
| Monitoring | Track classification accuracy, reply acceptance rate, and response times |

## How to Improve This Project

1. **Better classification**: Fine-tune ModernBERT on a labeled email intent dataset (e.g., TREC-6, custom)
2. **LLM generation**: Replace templates with Qwen3-Instruct for fluent, context-aware replies
3. **More prototypes**: Add 10+ prototype sentences per intent category for better coverage
4. **Urgency model**: Train a separate classifier for urgency instead of keyword matching
5. **Real dataset**: Evaluate on the Enron email dataset or customer support ticket data
6. **Human evaluation**: Have users rate reply quality across tones for preference learning

## Key Takeaways

1. **Two-stage pipelines** (classify then generate) are more reliable than single-prompt approaches because classification output is reusable and testable
2. **Embedding similarity** to labeled prototypes is a simple but effective intent classifier that requires no training data beyond a few examples per category
3. **Structured output** (typed fields, confidence scores) is essential for production systems -- free-form text is unparseable
4. **Tone control** via templates or prompt parameters lets the same system serve different audiences and situations
5. **Urgency detection** adds actionable prioritization on top of intent classification
6. **Error analysis** with confidence scores and score margins reveals where the classifier struggles and guides improvement